In [ ]:
from openai import OpenAI
import os
os.environ["OPENAI_API_KEY"] = "token"
import json
import tiktoken
from bs4 import BeautifulSoup
from datetime import datetime

### Helper

In [3]:
intetion_def_dict = {
    "Idea Generation": "Formulate and record initial thoughts and concepts",
    "Idea Organization": "Select the most useful materials and demarcate those generated ideas in a visually formatted way.",
    "Section Planning": "Initially create sections and sub-level structures.",
    "Text Production": "Translate their ideas into full languages, either from the writers’ language or borrowed sentences from an external source.",
    "Object Insertion": "Insert visual claims of their arguments (e.g., figures, tables, equations, footnotes, lists)",
    "Citation Integration": "Incorporate bibliographic references into a document and systematically link these references using citation commands.",
    "Cross-reference": "Link different sections, figures, tables, or other elements within a document through referencing commands.",
    "Macro Insertion": "Incorporate predefined commands or packages into a LaTeX document to alter its formatting.",
    "Fluency": "Fluency Fix grammatical or syntactic errors in the text or LaTeX commands.",
    "Coherence": "Logically link (1) any of the two or multiple sentences within the same paragraph; (2) any two subsequent paragraphs; or (3) objects to be consistent as a whole.",
    "Clarity": "Improve the semantic relationships between texts to be more straightforward and concise.",
    "Structural": "Improve the flow of information by modifying the location of texts and objects.",
    "Linguistic Style": "Modify texts with the writer’s writing preferences regarding styles and word choices, etc.",
    "Scientific Accuracy": "Update or correct scientific evidence (e.g., numbers, equations) for more accurate claims.",
    "Visual Formatting": "Modify the stylistic formatting of texts, objects, and citations",
}

In [ ]:
def strip_code_fences(text: str) -> str:
    """
    Remove Markdown code fences like ```latex ... ``` from model output.
    """
    # Remove starting and ending triple backticks with optional language tag
    if text.startswith("```"):
        # split into lines, drop first and last if they're code fences
        lines = text.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        return "\n".join(lines)
    return text

### Generate

In [6]:
def prompt_format(rec):
    
    label_text = ""
    for label in rec['label_list']:
        if label in intetion_def_dict:
            label_text += f"- {label}: {intetion_def_dict[label]}\n"
        
    SYSTEM_PROMPT = """
You are an expert writing assistant. 
Your role is to improve user-provided text according to specific criteria. 
The input will be LaTeX text. 
You may freely modify, remove, or add LaTeX commands, math notation, labels, references, or environments 
if it improves fluency, clarity, organization, or overall quality. 
Always ensure the output is valid LaTeX that can compile without errors. 
Always output only the improved LaTeX text, with no explanations or notes.
"""

    USER_PROMPT = f"""
Improve the following LaTeX text according to these criteria:
{label_text}
Important:
- You may freely modify LaTeX commands, math notation, labels, references, and environments.
- You may add new LaTeX text (e.g., \paragraph\{{}}, \section{{}}, or plain prose within environments) if needed.
- The final result must remain valid LaTeX.

Text to improve:
<<<TEXT_START
{rec['before_text']}
TEXT_END>>>
"""


    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}, 
        {"role": "user", "content": USER_PROMPT}
    ]
    return messages
    

In [ ]:
model = "gpt-5"
for i in [5]:
    with open(f"session/project_{i}_session.jsonl", "r") as f:
        data = [json.loads(line) for line in f]
    
    for rec in data:
        if rec['duration'] >= 1 and rec['index'] > 1979:
            output_file = f"session_experiment/generated_output/{model}/{i}/{rec['index']}.txt"
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            
            messages = prompt_format(rec)
            # print(messages)
            client = OpenAI()

            completion = client.chat.completions.create(
                model=model,
                messages=messages
            )

            generated_text = completion.choices[0].message.content

            with open(output_file, "w") as f:
                f.write(generated_text)

        